#### 基本用法

可以定义一个 InMemoryStore 来在线程之间存储用户信息。

In [1]:
import uuid
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [2]:
user_id = "1"
namespace_for_memory = (user_id, "memories") # 记忆通过 tuple 进行命名空间划分

memory_id = str(uuid.uuid4())
memory = {"food_preference" : "I like pizza"}

store.put(namespace_for_memory, memory_id, memory)

In [3]:
store.get(namespace_for_memory, memory_id)

Item(namespace=['1', 'memories'], key='d30dce26-d0c3-4200-82fe-1983f3e26b05', value={'food_preference': 'I like pizza'}, created_at='2026-05-01T13:32:22.636762+00:00', updated_at='2026-05-01T13:32:22.636765+00:00')

In [4]:
store.get(namespace_for_memory, memory_id).dict() # type: ignore

{'namespace': ['1', 'memories'],
 'key': 'd30dce26-d0c3-4200-82fe-1983f3e26b05',
 'value': {'food_preference': 'I like pizza'},
 'created_at': '2026-05-01T13:32:22.636762+00:00',
 'updated_at': '2026-05-01T13:32:22.636765+00:00'}

可以使用 store.search 方法读取命名空间中的记忆，该方法会将指定用户的全部记忆作为列表返回。列表中的最后一项即为最新的记忆。

In [5]:
memories = store.search(namespace_for_memory)
item = memories[-1]
item.dict()

{'namespace': ['1', 'memories'],
 'key': 'd30dce26-d0c3-4200-82fe-1983f3e26b05',
 'value': {'food_preference': 'I like pizza'},
 'created_at': '2026-05-01T13:32:22.636762+00:00',
 'updated_at': '2026-05-01T13:32:22.636765+00:00',
 'score': None}

#### 语义搜索

除了简单的检索功能外，该存储库还支持语义搜索，允许根据含义而非精确匹配来查找记忆。

In [6]:
from langchain.embeddings import init_embeddings
from agent_lab.model_hub import EMBEDDING_SMALL

store = InMemoryStore(
    index={
        'embed': init_embeddings(**EMBEDDING_SMALL),  # Embedding provider
        'dims': 1536,  # Embedding dimensions
        'fields': ['food_preference', '$'],  # Fields to embed
    }
)

通过配置 fields 参数或在存储记忆时指定 index 参数来控制哪些部分的记忆会被嵌入：

In [7]:
# Store with specific fields to embed
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {
        "food_preference": "I love Italian cuisine",
        "context": "Discussing dinner plans"
    },
    index=["food_preference"]  # Only embed "food_preferences" field
)

# Store without embedding (still retrievable, but not searchable)
store.put(
    namespace_for_memory,
    str(uuid.uuid4()),
    {"system_info": "Last updated: 2024-01-01"},
    index=False
)

现在进行搜索时，可以使用自然语言查询来查找相关记忆：

In [8]:
# Find memories about food preferences
# (This can be done after putting memories into the store)
memories = store.search(
    namespace_for_memory,
    query="What does the user like to eat?",
    limit=1  # Return top 3 matches
)

In [9]:
memories[0].dict()

{'namespace': ['1', 'memories'],
 'key': '162f180b-84fe-4713-ba27-1ddfe9b66947',
 'value': {'food_preference': 'I love Italian cuisine',
  'context': 'Discussing dinner plans'},
 'created_at': '2026-05-01T13:32:25.286357+00:00',
 'updated_at': '2026-05-01T13:32:25.286363+00:00',
 'score': 0.37711067104457946}